# Analisi di immagini utilizzando OpenCV

Scriviamo un piccolo programma che data un'immagine, ne evidenzi le parti che corrispondono a cani, gatti, o persone. Ovviamente per fare questo dovremo usare dei metodi di machine learning. In particolare adoperiamo la libreria `opencv` (disponibile anche per python) che permette di usare reti neurali per la classificazione.

Iniziamo installando il modulo `opencv-python`. Potete eseguire il comando da terminale oppure da Jupyterlab.

In [1]:
!pip install opencv-python

In [2]:
# Verify the install worked
import cv2
print("OpenCV version:", cv2.__version__)

OpenCV version: 4.13.0


## Mostriamo l'immagine per riferimento

In [3]:
image = cv2.imread("demophoto.jpg")

if image is None:
    print("Immagine non trovata!")
else:
    print("Loaded! Shape:", image.shape)

# Show it in a window
cv2.imshow("My Image", image)
cv2.waitKey(0)     # press any key to close
cv2.destroyAllWindows()

Loaded! Shape: (682, 1023, 3)


QFontDatabase: Cannot find font directory /home/massimo/.local/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/massimo/.local/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/massimo/.local/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/massimo/.local/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/massimo/.local/lib/python3.13/si

## Formato dell'immagine

Attenzione OpenCV usa un formato BGR invece di RGB.

In [17]:
print(image[0,0])

[ 16 134  75]


## Classificazione

Utilizzeremo la rete neurale [YoloV3](https://pjreddie.com/darknet/yolo/). Scarichiamo 

- i pesi della rete neurali
- un file di configurazione
- i nomi delle classi di oggetti da riconoscere.

In [18]:
!wget -nc https://pjreddie.com/media/files/yolov3.weights
!wget -nc https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg
!wget -nc https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names


File ‘yolov3.weights’ already there; not retrieving.

File ‘yolov3.cfg’ already there; not retrieving.

File ‘coco.names’ already there; not retrieving.



Con questi elementi possiamo dire a `opencv` di classificare gli oggetti nell'immagine e trasformare le etichette numeriche in nomi da visualizzare.

## Carichiamo il modello (la rete neurale)

In [19]:
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

print(f"Loaded {len(classes)} classes")

Loaded 80 classes


In [20]:
CLASSES  = {"person": (255,100,50), "cat": (50,200,50), "dog": (50,100,255)}

## Passiamo i dati nel modello

Otteniamo la classificazione dalla rete neurale.

In [25]:
ids[i]

np.int64(0)

In [22]:
import numpy as np

h, w, _ = image.shape

blob    = cv2.dnn.blobFromImage(image, 1/255, (416,416), swapRB=True)
net.setInput(blob)
outputs = net.forward(net.getUnconnectedOutLayersNames())

boxes, confs, ids = [], [], []
for out in outputs:
    for det in out:
        sc = det[5:]
        cid = np.argmax(sc)
        cf = sc[cid]
        if cf > 0.5 and classes[cid] in CLASSES:
            cx,cy,bw,bh = (det[:4]*[w,h,w,h]).astype("int")
            boxes.append([cx-bw//2, cy-bh//2, bw, bh])
            confs.append(float(cf))
            ids.append(cid)

idx = cv2.dnn.NMSBoxes(boxes, confs, 0.5, 0.4)

## Disegna l'immagine

Gli oggetti trovati dal modello vengono evidenziati nell'immagine. Eliminando l'ultimo commento diventerà

In [23]:
for i in idx.flatten():
    x,y,bw,bh = boxes[i]; lbl = classes[ids[i]]
    col = CLASSES[lbl]; txt = f"{lbl}: {confs[i]:.0%}"
    cv2.rectangle(image, (x,y), (x+bw,y+bh), col, 2)
    cv2.rectangle(image, (x,y-25), (x+130,y), col, -1)
    cv2.putText(image, txt, (x+4,y-7), cv2.FONT_HERSHEY_SIMPLEX,
               0.55, (255,255,255), 2)
    print(f"Found {lbl} ({confs[i]:.0%} confidence)")

cv2.imshow("Detections", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# cv2.imwrite("classified.jpg", image)

Found person (100% confidence)
Found person (99% confidence)
Found person (96% confidence)
Found person (57% confidence)
